# Unified MCTS — Phase A (value pretrain) — production warm-start

**Self-contained.** Warm-starts the board encoder + value head from production
**`mini_value_6o4.pt`** (384/6/6), leaves CLS / castle / EP fresh, then adapts for
**30k** iterations on Lichess SF value memmaps.

Production checkpoints are **read-only**.

## Upload

### A) Value memmaps
From `unified_mcts/data_export/value/`:
- `value_X.u8.memmap`, `value_R.u8.memmap`, `value_Y.f16.memmap`, `value_meta.json`

### B) Production value
- `value_transformer/mini_value_6o4.pt` (upload as `mini_value_6o4.pt`)

Attach both. Enable **GPU T4x2**. Run top → bottom.
Outputs: `/kaggle/working/unified_phase_a.pt` and `unified_phase_a_best.pt`.


In [ ]:
%%writefile model.py

"""

JointChessNet: board encoder (CLS) + FiLM-conditioned UCI decoder + policy/value heads.

"""

from __future__ import annotations



import math

from typing import Dict, Optional, Tuple



import torch

import torch.nn as nn

import torch.nn.functional as F



def _apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:

    x_even = x[..., ::2]

    x_odd = x[..., 1::2]

    cos = cos[None, None, :, :]

    sin = sin[None, None, :, :]

    out_even = x_even * cos - x_odd * sin

    out_odd = x_even * sin + x_odd * cos

    return torch.stack((out_even, out_odd), dim=-1).flatten(-2)





class BoardSelfAttention(nn.Module):

    def __init__(self, n_embd: int, n_head: int, dropout: float):

        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head

        self.n_embd = n_embd

        self.dropout = dropout

        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)

        self.c_proj = nn.Linear(n_embd, n_embd)

        self.attn_dropout = nn.Dropout(dropout)

        self.resid_dropout = nn.Dropout(dropout)

        self.flash = hasattr(F, "scaled_dot_product_attention")



    def forward(self, x: torch.Tensor) -> torch.Tensor:

        B, T, C = x.size()

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:

            y = F.scaled_dot_product_attention(

                q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=False

            )

        else:

            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))

            att = F.softmax(att, dim=-1)

            att = self.attn_dropout(att)

            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.resid_dropout(self.c_proj(y))





class FeedForward(nn.Module):

    def __init__(self, n_embd: int, dropout: float):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(n_embd, 4 * n_embd),

            nn.GELU(),

            nn.Linear(4 * n_embd, n_embd),

            nn.Dropout(dropout),

        )



    def forward(self, x: torch.Tensor) -> torch.Tensor:

        return self.net(x)





class BoardEncoderBlock(nn.Module):

    def __init__(self, n_embd: int, n_head: int, dropout: float):

        super().__init__()

        self.ln1 = nn.LayerNorm(n_embd)

        self.sa = BoardSelfAttention(n_embd, n_head, dropout)

        self.ln2 = nn.LayerNorm(n_embd)

        self.ff = FeedForward(n_embd, dropout)



    def forward(self, x: torch.Tensor) -> torch.Tensor:

        x = x + self.sa(self.ln1(x))

        x = x + self.ff(self.ln2(x))

        return x





class BoardEncoder(nn.Module):

    def __init__(

        self,

        n_embd: int = 384,

        n_head: int = 6,

        n_layer: int = 6,

        dropout: float = 0.1,

        n_rights_tokens: int = 2,

    ):

        super().__init__()

        self.n_embd = n_embd

        self.piece_emb = nn.Embedding(13, n_embd)

        self.rank_emb = nn.Embedding(8, n_embd)

        self.file_emb = nn.Embedding(8, n_embd)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, n_embd))

        self.castle_emb = nn.Embedding(16, n_embd)

        self.ep_emb = nn.Embedding(9, n_embd)

        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList(

            [BoardEncoderBlock(n_embd, n_head, dropout) for _ in range(n_layer)]

        )

        self.ln_f = nn.LayerNorm(n_embd)

        nn.init.normal_(self.cls_token, mean=0.0, std=0.02)



    def forward(self, board: torch.Tensor, rights: torch.Tensor) -> torch.Tensor:

        """

        board: [B, 64] long

        rights: [B, 8] long/byte

        returns CLS summary [B, d_b]

        """

        B = board.size(0)

        device = board.device

        ranks = torch.arange(64, device=device) // 8

        files = torch.arange(64, device=device) % 8

        ranks = ranks.unsqueeze(0).expand(B, -1)

        files = files.unsqueeze(0).expand(B, -1)



        e = self.piece_emb(board) + self.rank_emb(ranks) + self.file_emb(files)



        castle_ids = (

            rights[:, 0]

            + 2 * rights[:, 1]

            + 4 * rights[:, 2]

            + 8 * rights[:, 3]

        ).long()

        ep_ids = torch.where(

            rights[:, 4] > 0, 1 + rights[:, 5].long(), torch.zeros(B, device=device, dtype=torch.long)

        )



        cls = self.cls_token.expand(B, -1, -1)

        castle_tok = self.castle_emb(castle_ids).unsqueeze(1)

        ep_tok = self.ep_emb(ep_ids).unsqueeze(1)

        x = torch.cat([cls, e, castle_tok, ep_tok], dim=1)

        x = self.drop(x)

        for block in self.blocks:

            x = block(x)

        x = self.ln_f(x)

        return x[:, 0]





class CausalSelfAttention(nn.Module):

    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):

        super().__init__()

        assert n_embd % n_head == 0

        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)

        self.c_proj = nn.Linear(n_embd, n_embd)

        self.attn_dropout = nn.Dropout(dropout)

        self.resid_dropout = nn.Dropout(dropout)

        self.n_head = n_head

        self.n_embd = n_embd

        self.dropout = dropout

        head_dim = n_embd // n_head

        assert head_dim % 2 == 0

        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))

        self.register_buffer("_rope_inv_freq", inv_freq, persistent=False)

        self.register_buffer("_rope_cos_cached", torch.empty(0), persistent=False)

        self.register_buffer("_rope_sin_cached", torch.empty(0), persistent=False)

        self._rope_seq_len_cached = 0

        self._rope_cache_dtype = None

        self._rope_cache_device = None

        self.flash = hasattr(F, "scaled_dot_product_attention")

        if not self.flash:

            self.register_buffer(

                "bias", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)

            )



    def _rope_cos_sin(self, seq_len, device, dtype):

        if (

            self._rope_seq_len_cached >= seq_len

            and self._rope_cache_device == device

            and self._rope_cache_dtype == dtype

            and self._rope_cos_cached.numel() != 0

        ):

            return self._rope_cos_cached[:seq_len], self._rope_sin_cached[:seq_len]

        t = torch.arange(seq_len, device=device, dtype=torch.float32)

        freqs = torch.outer(t, self._rope_inv_freq.to(device=device))

        cos = freqs.cos().to(dtype=dtype)

        sin = freqs.sin().to(dtype=dtype)

        self._rope_cos_cached = cos

        self._rope_sin_cached = sin

        self._rope_seq_len_cached = seq_len

        self._rope_cache_device = device

        self._rope_cache_dtype = dtype

        return cos, sin



    def forward(self, x: torch.Tensor) -> torch.Tensor:

        B, T, C = x.size()

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        cos, sin = self._rope_cos_sin(T, q.device, q.dtype)

        q = _apply_rope(q, cos, sin)

        k = _apply_rope(k, cos, sin)

        if self.flash:

            y = F.scaled_dot_product_attention(

                q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True

            )

        else:

            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))

            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))

            att = F.softmax(att, dim=-1)

            att = self.attn_dropout(att)

            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.resid_dropout(self.c_proj(y))





class FiLM(nn.Module):

    def __init__(self, d_c: int, d: int):

        super().__init__()

        self.proj = nn.Linear(d_c, 2 * d)



    def forward(self, h: torch.Tensor, c: torch.Tensor) -> torch.Tensor:

        # h: [B,T,d], c: [B,d_c]

        gb = self.proj(c)

        gamma, beta = gb.chunk(2, dim=-1)

        gamma = gamma.unsqueeze(1)

        beta = beta.unsqueeze(1)

        return (1.0 + gamma) * h + beta





class DecoderBlock(nn.Module):

    def __init__(self, n_embd: int, n_head: int, block_size: int, d_c: int, dropout: float):

        super().__init__()

        self.ln1 = nn.LayerNorm(n_embd)

        self.film = FiLM(d_c, n_embd)

        self.sa = CausalSelfAttention(n_embd, n_head, block_size, dropout)

        self.ln2 = nn.LayerNorm(n_embd)

        self.ff = FeedForward(n_embd, dropout)



    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:

        h = self.film(self.ln1(x), c)

        x = x + self.sa(h)

        x = x + self.ff(self.ln2(x))

        return x





class ValueHead(nn.Module):

    def __init__(self, n_embd: int, dropout: float):

        super().__init__()

        self.drop = nn.Dropout(dropout)

        self.net = nn.Sequential(

            nn.Linear(n_embd, n_embd // 2),

            nn.GELU(),

            nn.Linear(n_embd // 2, 1),

            nn.Tanh(),

        )



    def forward(self, s: torch.Tensor) -> torch.Tensor:

        return self.net(self.drop(s)).squeeze(-1)





class JointChessNet(nn.Module):

    def __init__(

        self,

        vocab_size: int = 1793,

        block_size: int = 128,

        d_b: int = 384,

        n_head_b: int = 6,

        n_layer_b: int = 6,

        d: int = 500,

        n_head: int = 10,

        n_layer: int = 8,

        d_c: int | None = None,  # defaults to decoder width d

        dropout: float = 0.1,

    ):

        super().__init__()

        if d_c is None:

            d_c = d

        if d_c != d:

            raise ValueError(f"FiLM condition dim d_c={d_c} must match decoder embedding d={d}")

        self.vocab_size = vocab_size

        self.block_size = block_size

        self.d_c = d_c

        self.encoder = BoardEncoder(d_b, n_head_b, n_layer_b, dropout)

        self.cond_proj = nn.Linear(d_b, d_c)  # board summary → decoder width

        self.token_emb = nn.Embedding(vocab_size, d)

        self.blocks = nn.ModuleList(

            [DecoderBlock(d, n_head, block_size, d_c, dropout) for _ in range(n_layer)]

        )

        self.ln_f = nn.LayerNorm(d)

        self.lm_head = nn.Linear(d, vocab_size)

        self.value_head = ValueHead(d_b, dropout)

        self.apply(self._init_weights)



    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:

                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(module.weight, mean=0.0, std=0.02)



    def encode_board(self, board: torch.Tensor, rights: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:

        s = self.encoder(board.long(), rights.long())

        c = self.cond_proj(s)

        return s, c



    def forward_policy(

        self,

        hist: torch.Tensor,

        cond: torch.Tensor,

        targets: Optional[torch.Tensor] = None,

    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:

        """

        hist: [B, T]

        cond: [B, d_c]

        targets: [B] next-move ids (single-target) or None for logits only

        """

        B, T = hist.shape

        if T > self.block_size:

            hist = hist[:, -self.block_size :]

            T = self.block_size

        x = self.token_emb(hist.long())

        for block in self.blocks:

            x = block(x, cond)

        x = self.ln_f(x)

        logits = self.lm_head(x[:, -1, :])  # [B, V]

        loss = None

        if targets is not None:

            loss = F.cross_entropy(logits, targets.long())

        return logits, loss



    def forward_value(self, summary: torch.Tensor, y_v: Optional[torch.Tensor] = None):

        v = self.value_head(summary)

        loss = None

        if y_v is not None:

            loss = F.mse_loss(v, y_v.float())

        return v, loss



    def forward(

        self,

        hist: torch.Tensor,

        board: torch.Tensor,

        rights: torch.Tensor,

        y_pi: Optional[torch.Tensor] = None,

        y_v: Optional[torch.Tensor] = None,

        compute_policy: bool = True,

        compute_value: bool = True,

    ) -> Dict[str, Optional[torch.Tensor]]:

        s, c = self.encode_board(board, rights)

        out: Dict[str, Optional[torch.Tensor]] = {

            "summary": s,

            "cond": c,

            "logits": None,

            "v": None,

            "loss_pi": None,

            "loss_v": None,

        }

        if compute_policy:

            logits, loss_pi = self.forward_policy(hist, c, y_pi)

            out["logits"] = logits

            out["loss_pi"] = loss_pi

        if compute_value:

            v, loss_v = self.forward_value(s, y_v)

            out["v"] = v

            out["loss_v"] = loss_v

        return out



    def config_dict(self) -> dict:

        return {

            "vocab_size": self.vocab_size,

            "block_size": self.block_size,

            "d_b": self.encoder.n_embd,

            "n_head_b": self.encoder.blocks[0].sa.n_head,

            "n_layer_b": len(self.encoder.blocks),

            "d": self.token_emb.embedding_dim,

            "n_head": self.blocks[0].sa.n_head,

            "n_layer": len(self.blocks),

            "d_c": self.d_c,

        }



    def encoder_params(self):

        return list(self.encoder.parameters()) + list(self.cond_proj.parameters())



    def value_params(self):

        return list(self.value_head.parameters())



    def decoder_params(self):

        return (

            list(self.token_emb.parameters())

            + list(self.blocks.parameters())

            + list(self.ln_f.parameters())

            + list(self.lm_head.parameters())

        )



    def set_train_mode(self, mode: str):

        """

        Freeze schedules:

          phase_a / ft_value  — train encoder+value; freeze decoder

          phase_b / ft_policy — train decoder; freeze encoder+value

          all                 — everything trainable

        """

        if mode in ("phase_a", "ft_value"):

            set_requires_grad(self.encoder, True)

            set_requires_grad(self.cond_proj, False)

            set_requires_grad(self.value_head, True)

            set_requires_grad(self.token_emb, False)

            set_requires_grad(self.blocks, False)

            set_requires_grad(self.ln_f, False)

            set_requires_grad(self.lm_head, False)

        elif mode in ("phase_b", "ft_policy"):

            set_requires_grad(self.encoder, False)

            set_requires_grad(self.cond_proj, True)

            set_requires_grad(self.value_head, False)

            set_requires_grad(self.token_emb, True)

            set_requires_grad(self.blocks, True)

            set_requires_grad(self.ln_f, True)

            set_requires_grad(self.lm_head, True)

        elif mode == "all":

            for p in self.parameters():

                p.requires_grad = True

        else:

            raise ValueError(f"Unknown train mode: {mode}")





def set_requires_grad(module: nn.Module, flag: bool):

    for p in module.parameters():

        p.requires_grad = flag





def count_parameters(model: nn.Module) -> float:

    return sum(p.numel() for p in model.parameters()) / 1e6





def count_trainable(model: nn.Module) -> float:

    return sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6



In [ ]:
%%writefile dataset.py

"""Memmap datasets for value / policy training and FT mixes."""
from __future__ import annotations

import json
import os
from typing import Dict, Iterator, List, Optional, Sequence, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset, Sampler


def _open_memmap(path: str, dtype, shape, mode: str = "r"):
    return np.memmap(path, dtype=dtype, mode=mode, shape=tuple(shape))


class ValueMemmapDataset(Dataset):
    def __init__(self, data_dir: str, split: str = "train", val_ratio: float = 0.1, seed: int = 42):
        meta_path = os.path.join(data_dir, "value_meta.json")
        with open(meta_path) as f:
            self.meta = json.load(f)
        n = int(self.meta["N"])
        shape_x = (n, 64)
        shape_r = (n, int(self.meta.get("K", 8)))
        self.X = _open_memmap(os.path.join(data_dir, "value_X.u8.memmap"), np.uint8, shape_x)
        self.R = _open_memmap(os.path.join(data_dir, "value_R.u8.memmap"), np.uint8, shape_r)
        self.Y = _open_memmap(os.path.join(data_dir, "value_Y.f16.memmap"), np.float16, (n,))

        rng = np.random.RandomState(seed)
        idx = np.arange(n)
        rng.shuffle(idx)
        n_val = int(n * val_ratio)
        if split == "train":
            self.indices = idx[n_val:]
        else:
            self.indices = idx[:n_val]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int):
        j = int(self.indices[i])
        board = torch.from_numpy(np.array(self.X[j], dtype=np.int64))
        rights = torch.from_numpy(np.array(self.R[j], dtype=np.int64))
        y = torch.tensor(float(self.Y[j]), dtype=torch.float32)
        return board, rights, y


class PolicyMemmapDataset(Dataset):
    """Single corpus memmap. policy_yv is optional (zeros if missing)."""

    def __init__(self, data_dir: str, start: int = 0, end: Optional[int] = None):
        meta_path = os.path.join(data_dir, "policy_meta.json")
        with open(meta_path) as f:
            self.meta = json.load(f)
        n = int(self.meta["N"])
        self.block_size = int(self.meta.get("block_size", 128))
        self.K = int(self.meta.get("K", 8))
        end = n if end is None else end
        self.start = start
        self.end = end
        self.hist = _open_memmap(
            os.path.join(data_dir, "policy_hist.u16.memmap"),
            np.uint16,
            (n, self.block_size),
        )
        self.board = _open_memmap(
            os.path.join(data_dir, "policy_board.u8.memmap"), np.uint8, (n, 64)
        )
        self.rights = _open_memmap(
            os.path.join(data_dir, "policy_R.u8.memmap"), np.uint8, (n, self.K)
        )
        self.y = _open_memmap(
            os.path.join(data_dir, "policy_y.u16.memmap"), np.uint16, (n,)
        )
        yv_path = os.path.join(data_dir, "policy_yv.f16.memmap")
        if os.path.exists(yv_path):
            self.yv = _open_memmap(yv_path, np.float16, (n,))
            self.has_yv = True
        else:
            self.yv = None
            self.has_yv = False

    def __len__(self) -> int:
        return self.end - self.start

    def __getitem__(self, i: int):
        j = self.start + int(i)
        hist = torch.from_numpy(np.array(self.hist[j], dtype=np.int64))
        board = torch.from_numpy(np.array(self.board[j], dtype=np.int64))
        rights = torch.from_numpy(np.array(self.rights[j], dtype=np.int64))
        y = torch.tensor(int(self.y[j]), dtype=torch.long)
        if self.has_yv:
            yv = torch.tensor(float(self.yv[j]), dtype=torch.float32)
        else:
            yv = torch.tensor(0.0, dtype=torch.float32)
        return hist, board, rights, y, yv


class SourceMixSampler(Sampler[int]):
    """Sample global memmap indices with per-source mix weights (no O(N) weight tensor)."""

    def __init__(
        self,
        ranges: Sequence[Tuple[int, int]],
        weights: Sequence[float],
        num_samples: int,
        seed: int = 42,
    ):
        assert len(ranges) == len(weights) and len(ranges) > 0
        self.ranges = [(int(a), int(b)) for a, b in ranges]
        w = np.asarray(weights, dtype=np.float64)
        w = np.clip(w, 0.0, None)
        if w.sum() <= 0:
            w = np.ones_like(w)
        self.probs = w / w.sum()
        self.num_samples = int(num_samples)
        self.rng = np.random.RandomState(seed)

    def __iter__(self) -> Iterator[int]:
        for _ in range(self.num_samples):
            s = int(self.rng.choice(len(self.ranges), p=self.probs))
            start, end = self.ranges[s]
            yield int(self.rng.randint(start, end))

    def __len__(self) -> int:
        return self.num_samples


class MixedPolicyDataset(Dataset):
    """
    Policy corpus with 80/10/10 (or custom) source mix at sample time.
    meta.json sources: [{name, start, end}, ...]
    """

    def __init__(self, data_dir: str, mix: Optional[Dict[str, float]] = None):
        mix = mix or {"full": 0.80, "elite": 0.10, "weak": 0.10}
        meta_path = os.path.join(data_dir, "policy_meta.json")
        with open(meta_path) as f:
            self.meta = json.load(f)
        self.base = PolicyMemmapDataset(data_dir)
        sources = self.meta.get("sources")
        if not sources:
            sources = [{"name": "full", "start": 0, "end": int(self.meta["N"])}]

        self.ranges: List[Tuple[int, int]] = []
        self.source_weights: List[float] = []
        self.source_names: List[str] = []
        for src in sources:
            name = src["name"]
            w = float(mix.get(name, 0.0))
            start, end = int(src["start"]), int(src["end"])
            if w <= 0 or end <= start:
                continue
            self.ranges.append((start, end))
            self.source_weights.append(w)
            self.source_names.append(name)

        if not self.ranges:
            n = int(self.meta["N"])
            self.ranges = [(0, n)]
            self.source_weights = [1.0]
            self.source_names = ["all"]

        # Epoch length ≈ total rows (with replacement via sampler)
        self._len = int(self.meta["N"])

    def __len__(self) -> int:
        return self._len

    def __getitem__(self, j: int):
        """j is a global memmap index (from SourceMixSampler)."""
        j = int(j)
        hist = torch.from_numpy(np.array(self.base.hist[j], dtype=np.int64))
        board = torch.from_numpy(np.array(self.base.board[j], dtype=np.int64))
        rights = torch.from_numpy(np.array(self.base.rights[j], dtype=np.int64))
        y = torch.tensor(int(self.base.y[j]), dtype=torch.long)
        if self.base.has_yv:
            yv = torch.tensor(float(self.base.yv[j]), dtype=torch.float32)
        else:
            yv = torch.tensor(0.0, dtype=torch.float32)
        return hist, board, rights, y, yv

    def make_sampler(self, num_samples: Optional[int] = None, seed: int = 42) -> SourceMixSampler:
        n = num_samples or self._len
        return SourceMixSampler(self.ranges, self.source_weights, n, seed=seed)


class MixedValueFTDataset(Dataset):
    """
    Mix self-play value FT data with base Lichess value data.
    Returns (board, rights, y_v). Sample fraction `ft_frac` from FT, rest from base.
    """

    def __init__(
        self,
        ft_dir: str,
        base_dir: Optional[str] = None,
        ft_frac: float = 0.8,
        seed: int = 42,
    ):
        self.ft = ValueMemmapDataset(ft_dir, split="train", val_ratio=0.0, seed=seed)
        self.base = ValueMemmapDataset(base_dir, split="train", val_ratio=0.0, seed=seed) if base_dir else None
        self.ft_frac = float(ft_frac)
        self.rng = np.random.RandomState(seed)
        # Virtual length ~ FT size (with replacement sampling in __getitem__ via random)
        self._len = max(len(self.ft), 1)
        if self.base is not None:
            self._len = max(self._len, len(self.base))

    def __len__(self) -> int:
        return self._len

    def __getitem__(self, i: int):
        use_ft = self.base is None or (self.rng.rand() < self.ft_frac)
        ds = self.ft if use_ft else self.base
        j = int(self.rng.randint(0, len(ds)))
        return ds[j]


class MixedPolicyFTDataset(Dataset):
    """
    Mix self-play policy FT with base policy memmap.
    Returns (hist, board, rights, y_pi, y_v).
    """

    def __init__(
        self,
        ft_dir: str,
        base_dir: Optional[str] = None,
        ft_frac: float = 0.8,
        seed: int = 42,
    ):
        self.ft = PolicyMemmapDataset(ft_dir)
        self.base = PolicyMemmapDataset(base_dir) if base_dir else None
        self.ft_frac = float(ft_frac)
        self.rng = np.random.RandomState(seed)
        self._len = max(len(self.ft), 1)
        if self.base is not None:
            self._len = max(self._len, len(self.base))

    def __len__(self) -> int:
        return self._len

    def __getitem__(self, i: int):
        use_ft = self.base is None or (self.rng.rand() < self.ft_frac)
        ds = self.ft if use_ft else self.base
        j = int(self.rng.randint(0, len(ds)))
        return ds[j]


def write_memmap(path: str, arr: np.ndarray):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    mm = np.memmap(path, dtype=arr.dtype, mode="w+", shape=arr.shape)
    mm[:] = arr
    mm.flush()
    del mm


class StreamingPolicyWriter:
    """Append policy rows to raw memmap files without holding the full corpus in RAM."""

    def __init__(self, out_dir: str, block_size: int, rights_dim: int, flush_every: int = 50_000):
        self.out_dir = out_dir
        self.block_size = block_size
        self.rights_dim = rights_dim
        self.flush_every = flush_every
        os.makedirs(out_dir, exist_ok=True)
        self.paths = {
            "hist": os.path.join(out_dir, "policy_hist.u16.memmap"),
            "board": os.path.join(out_dir, "policy_board.u8.memmap"),
            "rights": os.path.join(out_dir, "policy_R.u8.memmap"),
            "y": os.path.join(out_dir, "policy_y.u16.memmap"),
            "yv": os.path.join(out_dir, "policy_yv.f16.memmap"),
        }
        for p in self.paths.values():
            open(p, "wb").close()
        self.n = 0
        self._hist: List[np.ndarray] = []
        self._board: List[np.ndarray] = []
        self._rights: List[np.ndarray] = []
        self._y: List[np.ndarray] = []
        self._yv: List[np.ndarray] = []

    def add(self, hist, board, rights, y, yv) -> None:
        self._hist.append(np.asarray(hist, dtype=np.uint16))
        self._board.append(np.asarray(board, dtype=np.uint8))
        self._rights.append(np.asarray(rights, dtype=np.uint8))
        self._y.append(np.asarray(y, dtype=np.uint16))
        self._yv.append(np.asarray(yv, dtype=np.float16))
        if len(self._y) >= self.flush_every:
            self.flush()

    def flush(self) -> None:
        if not self._y:
            return
        hist = np.stack(self._hist).astype(np.uint16, copy=False)
        board = np.stack(self._board).astype(np.uint8, copy=False)
        rights = np.stack(self._rights).astype(np.uint8, copy=False)
        y = np.asarray(self._y, dtype=np.uint16)
        yv = np.asarray(self._yv, dtype=np.float16)
        with open(self.paths["hist"], "ab") as f:
            f.write(hist.tobytes())
        with open(self.paths["board"], "ab") as f:
            f.write(board.tobytes())
        with open(self.paths["rights"], "ab") as f:
            f.write(rights.tobytes())
        with open(self.paths["y"], "ab") as f:
            f.write(y.tobytes())
        with open(self.paths["yv"], "ab") as f:
            f.write(yv.tobytes())
        self.n += len(y)
        self._hist.clear()
        self._board.clear()
        self._rights.clear()
        self._y.clear()
        self._yv.clear()

    def close(self) -> int:
        self.flush()
        return self.n


In [ ]:
%%writefile warmstart.py
"""
Warm-start helpers for the unified pipeline.

1) Production policy (vocab 4033) → vocab1 PolicyHead state (1793) via UCI-row remap.
2) Vocab1 PolicyHead → JointChessNet decoder (+ zero FiLM).
3) Production value (ChessFormer 384/6/6) → JointChessNet encoder + value head
   (CLS / castle / EP stay fresh).
"""
from __future__ import annotations

import pickle
from typing import Dict, List, Tuple

import torch
import torch.nn as nn


def _strip_orig_mod(state: dict) -> dict:
    return {k.removeprefix("_orig_mod."): v for k, v in state.items()}


def _load_state(ckpt_path: str) -> dict:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    if hasattr(state, "state_dict"):
        state = state.state_dict()
    return _strip_orig_mod(state)


def build_vocab1_policy_state(
    legacy_ckpt_path: str,
    vocab_old_path: str,
    vocab_new_path: str,
) -> Dict[str, torch.Tensor]:
    """
    Remap a production PolicyHead checkpoint (vocab.pkl / 4033) into a vocab1
    (1793) state dict. Every vocab1 token must exist in the old vocab.
    Transformer body (blocks / ln_f / position_embedding) is copied 1:1.
    """
    with open(vocab_old_path, "rb") as f:
        stoi_old = pickle.load(f)
    with open(vocab_new_path, "rb") as f:
        stoi_new = pickle.load(f)

    missing = [t for t in stoi_new if t not in stoi_old]
    if missing:
        raise ValueError(
            f"{len(missing)} vocab1 tokens missing from production vocab "
            f"(examples: {missing[:5]})"
        )

    state = _load_state(legacy_ckpt_path)
    emb_old = state["token_embedding_table.weight"]
    lm_w_old = state["lm_head.weight"]
    lm_b_old = state["lm_head.bias"]
    if emb_old.shape[0] != len(stoi_old):
        raise ValueError(
            f"Checkpoint vocab rows {emb_old.shape[0]} != len(vocab_old) {len(stoi_old)}"
        )

    d = emb_old.shape[1]
    v_new = len(stoi_new)
    # Build gather index: new_id -> old_id, sorted by new id
    itos_new = {int(i): s for s, i in stoi_new.items()}
    gather = torch.tensor(
        [int(stoi_old[itos_new[i]]) for i in range(v_new)],
        dtype=torch.long,
    )

    out: Dict[str, torch.Tensor] = {}
    for k, v in state.items():
        if k in (
            "token_embedding_table.weight",
            "lm_head.weight",
            "lm_head.bias",
        ):
            continue
        out[k] = v

    out["token_embedding_table.weight"] = emb_old.index_select(0, gather).contiguous()
    out["lm_head.weight"] = lm_w_old.index_select(0, gather).contiguous()
    out["lm_head.bias"] = lm_b_old.index_select(0, gather).contiguous()

    if out["token_embedding_table.weight"].shape != (v_new, d):
        raise ValueError(
            f"Remapped emb shape {tuple(out['token_embedding_table.weight'].shape)} "
            f"!= ({v_new}, {d})"
        )
    return out


def load_legacy_policy_decoder(model: nn.Module, ckpt_path: str, strict_vocab: bool = True) -> int:
    """
    Copy legacy PolicyHead weights into model's decoder and zero-init FiLM.
    Returns the number of tensors copied.
    """
    state = _load_state(ckpt_path)

    emb = state["token_embedding_table.weight"]
    if strict_vocab and emb.shape[0] != model.vocab_size:
        raise ValueError(
            f"Legacy vocab {emb.shape[0]} != model vocab {model.vocab_size}; "
            "expected a vocab1 (1793) checkpoint like ultra_v1_o1.pt"
        )
    if emb.shape[1] != model.token_emb.embedding_dim:
        raise ValueError(
            f"Legacy width {emb.shape[1]} != decoder width {model.token_emb.embedding_dim}"
        )

    key_map = {"token_embedding_table.weight": "token_emb.weight"}
    n_layers = len(model.blocks)
    for i in range(n_layers):
        for a, b in (
            (f"blocks.{i}.ln1.weight", f"blocks.{i}.ln1.weight"),
            (f"blocks.{i}.ln1.bias", f"blocks.{i}.ln1.bias"),
            (f"blocks.{i}.sa.c_attn.weight", f"blocks.{i}.sa.c_attn.weight"),
            (f"blocks.{i}.sa.c_proj.weight", f"blocks.{i}.sa.c_proj.weight"),
            (f"blocks.{i}.sa.c_proj.bias", f"blocks.{i}.sa.c_proj.bias"),
            (f"blocks.{i}.ln2.weight", f"blocks.{i}.ln2.weight"),
            (f"blocks.{i}.ln2.bias", f"blocks.{i}.ln2.bias"),
            (f"blocks.{i}.ffwd.net.0.weight", f"blocks.{i}.ff.net.0.weight"),
            (f"blocks.{i}.ffwd.net.0.bias", f"blocks.{i}.ff.net.0.bias"),
            (f"blocks.{i}.ffwd.net.2.weight", f"blocks.{i}.ff.net.2.weight"),
            (f"blocks.{i}.ffwd.net.2.bias", f"blocks.{i}.ff.net.2.bias"),
        ):
            key_map[a] = b
    key_map["ln_f.weight"] = "ln_f.weight"
    key_map["ln_f.bias"] = "ln_f.bias"
    key_map["lm_head.weight"] = "lm_head.weight"
    key_map["lm_head.bias"] = "lm_head.bias"

    own = model.state_dict()
    copied = 0
    for src, dst in key_map.items():
        if src not in state:
            raise KeyError(f"Legacy checkpoint missing key: {src}")
        if own[dst].shape != state[src].shape:
            raise ValueError(
                f"Shape mismatch {src} {tuple(state[src].shape)} vs {dst} {tuple(own[dst].shape)}"
            )
        own[dst] = state[src]
        copied += 1
    model.load_state_dict(own)

    # Zero FiLM so the warm-started decoder is exactly the legacy function at step 0.
    for block in model.blocks:
        nn.init.zeros_(block.film.proj.weight)
        nn.init.zeros_(block.film.proj.bias)
    return copied


def load_legacy_value_encoder(model: nn.Module, ckpt_path: str) -> int:
    """
    Copy production ChessFormer (384/6/6) weights into JointChessNet encoder + value head.
    Leaves cls_token / castle_emb / ep_emb at fresh init (unified-only tokens).
    Returns number of tensors copied.
    """
    state = _load_state(ckpt_path)
    own = model.state_dict()

    key_map = {
        "piece_embedding.weight": "encoder.piece_emb.weight",
        "rank_embedding.weight": "encoder.rank_emb.weight",
        "file_embedding.weight": "encoder.file_emb.weight",
        "ln_f.weight": "encoder.ln_f.weight",
        "ln_f.bias": "encoder.ln_f.bias",
        "value_head.net.0.weight": "value_head.net.0.weight",
        "value_head.net.0.bias": "value_head.net.0.bias",
        "value_head.net.2.weight": "value_head.net.2.weight",
        "value_head.net.2.bias": "value_head.net.2.bias",
    }
    n_layers = len(model.encoder.blocks)
    for i in range(n_layers):
        for a, b in (
            (f"blocks.{i}.ln1.weight", f"encoder.blocks.{i}.ln1.weight"),
            (f"blocks.{i}.ln1.bias", f"encoder.blocks.{i}.ln1.bias"),
            (f"blocks.{i}.sa.c_attn.weight", f"encoder.blocks.{i}.sa.c_attn.weight"),
            (f"blocks.{i}.sa.c_proj.weight", f"encoder.blocks.{i}.sa.c_proj.weight"),
            (f"blocks.{i}.sa.c_proj.bias", f"encoder.blocks.{i}.sa.c_proj.bias"),
            (f"blocks.{i}.ln2.weight", f"encoder.blocks.{i}.ln2.weight"),
            (f"blocks.{i}.ln2.bias", f"encoder.blocks.{i}.ln2.bias"),
            (f"blocks.{i}.ff.net.0.weight", f"encoder.blocks.{i}.ff.net.0.weight"),
            (f"blocks.{i}.ff.net.0.bias", f"encoder.blocks.{i}.ff.net.0.bias"),
            (f"blocks.{i}.ff.net.2.weight", f"encoder.blocks.{i}.ff.net.2.weight"),
            (f"blocks.{i}.ff.net.2.bias", f"encoder.blocks.{i}.ff.net.2.bias"),
        ):
            key_map[a] = b

    copied = 0
    for src, dst in key_map.items():
        if src not in state:
            raise KeyError(f"Legacy value checkpoint missing key: {src}")
        if own[dst].shape != state[src].shape:
            raise ValueError(
                f"Shape mismatch {src} {tuple(state[src].shape)} vs {dst} {tuple(own[dst].shape)}"
            )
        own[dst] = state[src]
        copied += 1
    model.load_state_dict(own)
    return copied


def fresh_and_pretrained_param_groups(model: nn.Module):
    """
    Split trainable Phase B params for two learning rates:
    fresh (FiLM + cond_proj, random/zero init) vs pretrained legacy decoder weights.
    """
    fresh, pretrained = [], []
    fresh += [p for p in model.cond_proj.parameters() if p.requires_grad]
    for block in model.blocks:
        fresh += [p for p in block.film.proj.parameters() if p.requires_grad]
        for mod in (block.ln1, block.sa, block.ln2, block.ff):
            pretrained += [p for p in mod.parameters() if p.requires_grad]
    for mod in (model.token_emb, model.ln_f, model.lm_head):
        pretrained += [p for p in mod.parameters() if p.requires_grad]
    return fresh, pretrained


def fresh_and_pretrained_value_groups(model: nn.Module) -> Tuple[List[nn.Parameter], List[nn.Parameter]]:
    """
    Phase A dual LR: fresh = CLS / castle / EP embeds; pretrained = rest of encoder+value.
    """
    fresh_ids = {
        id(model.encoder.cls_token),
        *[id(p) for p in model.encoder.castle_emb.parameters()],
        *[id(p) for p in model.encoder.ep_emb.parameters()],
    }
    fresh, pretrained = [], []
    for p in model.parameters():
        if not p.requires_grad:
            continue
        if id(p) in fresh_ids:
            fresh.append(p)
        else:
            pretrained.append(p)
    return fresh, pretrained


## Find data


In [ ]:
import os

input_root = "/kaggle/input"

DATA_DIR = None
LEGACY_VALUE = None

for root, dirs, files in os.walk(input_root):
    if DATA_DIR is None and "value_meta.json" in files and "value_X.u8.memmap" in files:
        DATA_DIR = root
        print("DATA_DIR", DATA_DIR)
    if LEGACY_VALUE is None and "mini_value_6o4.pt" in files:
        LEGACY_VALUE = os.path.join(root, "mini_value_6o4.pt")
        print("LEGACY_VALUE", LEGACY_VALUE)

assert DATA_DIR, "Attach the value memmap dataset (needs value_meta.json + value_*.memmap)"
assert LEGACY_VALUE, "Upload production mini_value_6o4.pt (read-only warm-start for encoder+value)"


## Train Phase A


In [ ]:
import math
import time

import torch
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dataset import ValueMemmapDataset
from model import JointChessNet, count_parameters, count_trainable
from warmstart import fresh_and_pretrained_value_groups, load_legacy_value_encoder

OUT = "/kaggle/working/unified_phase_a.pt"
OUT_BEST = "/kaggle/working/unified_phase_a_best.pt"
BATCH = 1024
MAX_ITERS = 30_000
LR_FRESH = 3e-4       # CLS / castle / EP
LR_PRETRAINED = 1e-4  # production encoder + value head
WARMUP = 2000
EVAL_EVERY = 1000
SAVE_EVERY = 2000
LOG_EVERY = 50

device = torch.device("cuda")
print("GPU", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

train_ds = ValueMemmapDataset(DATA_DIR, split="train")
val_ds = ValueMemmapDataset(DATA_DIR, split="val")
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)

model = JointChessNet(dropout=0.1).to(device)
n_copied = load_legacy_value_encoder(model, LEGACY_VALUE)
print(f"warm-started encoder+value from {LEGACY_VALUE} ({n_copied} tensors; CLS/castle/EP fresh)")
model.set_train_mode("phase_a")
params = [p for p in model.parameters() if p.requires_grad]
fresh, pretrained = fresh_and_pretrained_value_groups(model)
opt = torch.optim.AdamW(
    [
        {"params": fresh, "lr": LR_FRESH, "base_lr": LR_FRESH},
        {"params": pretrained, "lr": LR_PRETRAINED, "base_lr": LR_PRETRAINED},
    ],
    weight_decay=0.01,
)
scaler = GradScaler(enabled=True)

def save_checkpoint(obj, path):
    # Legacy pickle .pt (default torch.save is zip; Kaggle shows it as .zip)
    torch.save(obj, path, _use_new_zipfile_serialization=False)

print(
    f"trainable={count_trainable(model):.2f}M / total={count_parameters(model):.2f}M  "
    f"train_n={len(train_ds):,}  fresh={sum(p.numel() for p in fresh)/1e6:.3f}M"
)


def get_lr(it, base):
    if it < WARMUP:
        return base * (it + 1) / WARMUP
    ratio = (it - WARMUP) / max(1, MAX_ITERS - WARMUP)
    return base / 10 + 0.5 * (1 + math.cos(math.pi * ratio)) * (base - base / 10)


# Step-0 MSE sanity (warm body should beat random ~0.33 quickly)
model.eval()
with torch.no_grad():
    board0, rights0, yv0 = next(iter(train_loader))
    board0, rights0, yv0 = board0.to(device), rights0.to(device), yv0.to(device)
    hist0 = torch.zeros(board0.size(0), model.block_size, dtype=torch.long, device=device)
    with autocast(device_type="cuda"):
        out0 = model(hist0, board0, rights0, y_v=yv0, compute_policy=False, compute_value=True)
    mse0 = float(out0["loss_v"])
model.train()
print(f"Step-0 train MSE (1 batch): {mse0:.5f}")
# Production value mean-pools 64 squares; unified reads a fresh CLS (+ castle/EP).
# Copied body helps, but step-0 CLS value can be far off until Phase A adapts.
# Only fail if something is catastrophically wrong (NaN / absurd).
assert mse0 == mse0 and mse0 < 5.0, f"Warm-start looks broken: MSE={mse0:.5f}"
if mse0 > 0.5:
    print(
        "Note: step-0 MSE > 0.5 is expected with CLS vs mean-pool mismatch; "
        "continue — Phase A adapts CLS/rights."
    )

it = 0
best = 1e9
train_iter = iter(train_loader)
pbar = tqdm(total=MAX_ITERS, desc="Training", unit="iter", dynamic_ncols=True)
t0 = time.time()
while it < MAX_ITERS:
    for g in opt.param_groups:
        g["lr"] = get_lr(it, g["base_lr"])
    lr = opt.param_groups[0]["lr"]
    try:
        board, rights, yv = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        board, rights, yv = next(train_iter)
    board, rights, yv = board.to(device), rights.to(device), yv.to(device)

    opt.zero_grad(set_to_none=True)
    with autocast(device_type="cuda"):
        hist = torch.zeros(board.size(0), model.block_size, dtype=torch.long, device=device)
        out = model(hist, board, rights, y_v=yv, compute_policy=False, compute_value=True)
        loss = out["loss_v"]
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(opt)
    scaler.update()

    loss_f = float(loss.detach())
    pbar.set_description(f"Iter {it} | MSE {loss_f:.5f} | LR {lr:.2e}")
    if it % LOG_EVERY == 0:
        elapsed = time.time() - t0
        rate = (it + 1) / max(elapsed, 1e-6)
        eta_s = (MAX_ITERS - it - 1) / max(rate, 1e-6)
        eta_m, eta_sec = divmod(int(eta_s), 60)
        eta_h, eta_m = divmod(eta_m, 60)
        print(
            f"step {it}: train loss {loss_f:.5f}, lr {lr:.2e}, "
            f"{rate:.1f} it/s, remaining {eta_h:d}:{eta_m:02d}:{eta_sec:02d}",
            flush=True,
        )
    if it > 0 and it % EVAL_EVERY == 0:
        model.eval()
        losses = []
        with torch.no_grad():
            for board, rights, yv in val_loader:
                board, rights, yv = board.to(device), rights.to(device), yv.to(device)
                hist = torch.zeros(board.size(0), model.block_size, dtype=torch.long, device=device)
                out = model(hist, board, rights, y_v=yv, compute_policy=False, compute_value=True)
                losses.append(float(out["loss_v"]))
                if len(losses) >= 20:
                    break
        val = sum(losses) / len(losses)
        print(f"\niter {it}: val_mse={val:.5f}", flush=True)
        if val < best:
            best = val
            save_checkpoint(
                {"model": model.state_dict(), "config": model.config_dict(), "iter": it, "best_val": best},
                OUT_BEST,
            )
            print("saved best", OUT_BEST, flush=True)
        model.train()

    if it > 0 and it % SAVE_EVERY == 0:
        save_checkpoint(
            {"model": model.state_dict(), "config": model.config_dict(), "iter": it, "best_val": best},
            OUT,
        )
        print("saved", OUT, "iter", it, flush=True)
    it += 1
    pbar.update(1)

pbar.close()
save_checkpoint(
    {"model": model.state_dict(), "config": model.config_dict(), "iter": it, "best_val": best},
    OUT,
)
print("done", OUT, "best_val", best, flush=True)
